# Phase 2: Reasoning Tasks Feasibility Check

Before building the distillation loop, we must verify that our chosen model (`SmolLM2-1.7B-Instruct`) actually exhibits the required behavior on our reasoning tasks:
1. **Teacher Competence**: The model must be able to solve the task when given 4 demonstrations.
2. **Student Ignorance**: The model must fail to solve the task when given 0 demonstrations (0-shot).

If the teacher fails, there is no signal to distill. If the 0-shot student succeeds, there is no gap to close.
We will evaluate all 20 tasks from `forsmol.json` and `forsmol2.json` to find the tasks with the strongest distillation potential.

In [8]:
!pip install transformers torch

## Step 1 — Setup and Load Model

In [9]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained(model_name)

# We only need one model for this feasibility check
if 'model' not in globals():
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",
    ).to(device)
model.eval()

print(f"Model {model_name} loaded in bfloat16.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Model HuggingFaceTB/SmolLM2-1.7B-Instruct loaded in bfloat16.


## Step 2 — Load Datasets

In [14]:
import json
raw_tasks_json = r"""[
  {
    "id": 1,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nAnswer: Bob\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nAnswer: Tom\n\nExample 3\nJohn has the book.\nSarah has the pen.\nSwap John and Sarah.\nWho has the book?\nAnswer: Sarah\n\nExample 4\nMike has the watch.\nLucy has the phone.\nSwap Mike and Lucy.\nWho has the phone?\nAnswer: Mike\n\nQuestion\nDavid has the laptop.\nAnna has the tablet.\nSwap David and Anna.\nWho has the laptop?\nOptions:\nA) Alice\nB) Anna\nC) David\nD) Luke\nAnswer:",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nDavid has the laptop.\nAnna has the tablet.\nSwap David and Anna.\nWho has the laptop?\nOptions:\nA) Alice\nB) Anna\nC) David\nD) Luke\nAnswer:",
    "answer": "B"
  },
  {
    "id": 2,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nAnswer: Charlie\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nAnswer: Sam\n\nExample 3\nAmy has the hat.\nBen has the bag.\nCarl has the map.\nSwap Ben and Carl.\nSwap Amy and Ben.\nWho has the hat?\nAnswer: Carl\n\nExample 4\nJack has the phone.\nKate has the watch.\nLuke has the wallet.\nSwap Jack and Luke.\nSwap Kate and Luke.\nWho has the watch?\nAnswer: Jack\n\nQuestion\nJohn has the laptop.\nMary has the tablet.\nPeter has the camera.\nSwap John and Peter.\nSwap Mary and Peter.\nWho has the laptop?\nOptions:\nA) Anna\nB) Emma\nC) Mary\nD) Peter\nAnswer:",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nJohn has the laptop.\nMary has the tablet.\nPeter has the camera.\nSwap John and Peter.\nSwap Mary and Peter.\nWho has the laptop?\nOptions:\nA) Anna\nB) Emma\nC) Mary\nD) Peter\nAnswer:",
    "answer": "C"
  },
  {
    "id": 3,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nAnswer: (3,2)\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nAnswer: (2,0)\n\nExample 3\nStart (1,5)\nEast 4\nSouth 2\nAnswer: (5,3)\n\nExample 4\nStart (3,3)\nNorth 1\nWest 3\nAnswer: (0,4)\n\nQuestion\nStart (2,2)\nNorth 3\nEast 4\nSouth 1\nOptions:\nA) (0,4)\nB) (0,6)\nC) (3,2)\nD) (6,4)\nAnswer:",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (2,2)\nNorth 3\nEast 4\nSouth 1\nOptions:\nA) (0,4)\nB) (0,6)\nC) (3,2)\nD) (6,4)\nAnswer:",
    "answer": "D"
  },
  {
    "id": 4,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final direction.\n\nExample 1\nFace North.\nTurn Right.\nTurn Right.\nAnswer: South\n\nExample 2\nFace East.\nTurn Left.\nAnswer: North\n\nExample 3\nFace South.\nTurn Left.\nTurn Left.\nAnswer: North\n\nExample 4\nFace West.\nTurn Right.\nAnswer: North\n\nQuestion\nFace North.\nTurn Left.\nTurn Right.\nTurn Right.\nOptions:\nA) East\nB) North\nC) South\nD) West\nAnswer:",
    "student_prompt": "Task: Determine the final direction.\n\nQuestion\nFace North.\nTurn Left.\nTurn Right.\nTurn Right.\nOptions:\nA) East\nB) North\nC) South\nD) West\nAnswer:",
    "answer": "A"
  },
  {
    "id": 5,
    "task": "formal_fallacies",
    "teacher_prompt": "Task: Decide whether the argument is Valid or Invalid.\n\nExample 1\nAll cats are mammals.\nTom is a cat.\nTherefore Tom is a mammal.\nAnswer: Valid\n\nExample 2\nAll birds fly.\nTweety flies.\nTherefore Tweety is a bird.\nAnswer: Invalid\n\nExample 3\nAll apples are fruits.\nThis is a fruit.\nTherefore this is an apple.\nAnswer: Invalid\n\nExample 4\nAll doctors studied medicine.\nAlice is a doctor.\nTherefore Alice studied medicine.\nAnswer: Valid\n\nQuestion\nAll programmers know Python.\nJohn knows Python.\nTherefore John is a programmer.\nOptions:\nA) Invalid\nB) Valid\nAnswer:",
    "student_prompt": "Task: Decide whether the argument is Valid or Invalid.\n\nQuestion\nAll programmers know Python.\nJohn knows Python.\nTherefore John is a programmer.\nOptions:\nA) Invalid\nB) Valid\nAnswer:",
    "answer": "A"
  },
  {
    "id": 6,
    "task": "logical_deduction",
    "teacher_prompt": "Task: Answer Yes or No.\n\nExample 1\nEvery cat is an animal.\nEvery animal breathes.\nTom is a cat.\nDoes Tom breathe?\nAnswer: Yes\n\nExample 2\nEvery engineer knows math.\nEveryone who knows math can solve equations.\nAlice is an engineer.\nCan Alice solve equations?\nAnswer: Yes\n\nExample 3\nEvery whale is a mammal.\nEvery mammal is warm-blooded.\nBlue is a whale.\nIs Blue warm-blooded?\nAnswer: Yes\n\nExample 4\nEvery bird has feathers.\nEvery animal with feathers lays eggs.\nRobin is a bird.\nDoes Robin lay eggs?\nAnswer: Yes\n\nQuestion\nEvery square is a rectangle.\nEvery rectangle has four sides.\nShape A is a square.\nDoes Shape A have four sides?\nOptions:\nA) No\nB) Yes\nAnswer:",
    "student_prompt": "Task: Answer Yes or No.\n\nQuestion\nEvery square is a rectangle.\nEvery rectangle has four sides.\nShape A is a square.\nDoes Shape A have four sides?\nOptions:\nA) No\nB) Yes\nAnswer:",
    "answer": "B"
  },
  {
    "id": 7,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nAnswer: True\n\nExample 2\n(False OR False) AND True\nAnswer: False\n\nExample 3\n(True OR False) AND (True AND True)\nAnswer: True\n\nExample 4\n(False AND True) OR False\nAnswer: False\n\nQuestion\n((True OR False) AND False) OR (False OR True)\nOptions:\nA) False\nB) True\nAnswer:",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n((True OR False) AND False) OR (False OR True)\nOptions:\nA) False\nB) True\nAnswer:",
    "answer": "B"
  },
  {
    "id": 8,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nAnswer: 18\n\nExample 2\n10-(3*2)\nAnswer: 4\n\nExample 3\n24/6+7\nAnswer: 11\n\nExample 4\n(8-3)*(5-2)\nAnswer: 15\n\nQuestion\n((12/3)+5)*2\nOptions:\nA) 15\nB) 16\nC) 18\nD) 24\nAnswer:",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((12/3)+5)*2\nOptions:\nA) 15\nB) 16\nC) 18\nD) 24\nAnswer:",
    "answer": "C"
  },
  {
    "id": 9,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nAnswer: Balanced\n\nExample 2\n((())\nAnswer: Unbalanced\n\nExample 3\n(((())))\nAnswer: Balanced\n\nExample 4\n())(()\nAnswer: Unbalanced\n\nQuestion\n((())())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer:",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((())())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer:",
    "answer": "A"
  },
  {
    "id": 10,
    "task": "word_sorting",
    "teacher_prompt": "Task: Sort the words alphabetically.\n\nExample 1\npear apple banana\nAnswer: apple banana pear\n\nExample 2\ndog cat bird\nAnswer: bird cat dog\n\nExample 3\norange kiwi grape\nAnswer: grape kiwi orange\n\nExample 4\nzebra lion tiger\nAnswer: lion tiger zebra\n\nQuestion\nviolin apple monkey chair\nOptions:\nA) apple chair monkey violin\nB) chair monkey violin apple\nC) violin monkey chair apple\nD) violin monkey chair apple\nAnswer:",
    "student_prompt": "Task: Sort the words alphabetically.\n\nQuestion\nviolin apple monkey chair\nOptions:\nA) apple chair monkey violin\nB) chair monkey violin apple\nC) violin monkey chair apple\nD) violin monkey chair apple\nAnswer:",
    "answer": "A"
  },
  {
    "id": 11,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nCharlie has the green ball.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the blue ball?\nAnswer: Charlie\n\nExample 2\nTom has the key.\nEmma has the coin.\nRyan has the map.\nSwap Emma and Ryan.\nSwap Tom and Emma.\nWho has the map?\nAnswer: Tom\n\nExample 3\nAmy has the ring.\nBen has the watch.\nCarl has the wallet.\nSwap Ben and Carl.\nSwap Amy and Ben.\nWho has the ring?\nAnswer: Carl\n\nExample 4\nJack has the phone.\nKate has the tablet.\nLuke has the laptop.\nSwap Jack and Luke.\nSwap Luke and Kate.\nWho has the tablet?\nAnswer: Jack\n\nQuestion\nDavid has the notebook.\nAnna has the camera.\nPeter has the calculator.\nSwap David and Peter.\nSwap Anna and Peter.\nWho has the calculator?\nOptions:\nA) Alice\nB) Anna\nC) Carl\nD) Mary\nAnswer:",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nDavid has the notebook.\nAnna has the camera.\nPeter has the calculator.\nSwap David and Peter.\nSwap Anna and Peter.\nWho has the calculator?\nOptions:\nA) Alice\nB) Anna\nC) Carl\nD) Mary\nAnswer:",
    "answer": "B"
  },
  {
    "id": 12,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (2,3)\nNorth 4\nWest 2\nSouth 1\nAnswer: (0,6)\n\nExample 2\nStart (1,1)\nEast 5\nNorth 2\nWest 1\nAnswer: (5,3)\n\nExample 3\nStart (4,4)\nSouth 3\nEast 2\nAnswer: (6,1)\n\nExample 4\nStart (0,5)\nEast 2\nSouth 4\nWest 1\nAnswer: (1,1)\n\nQuestion\nStart (3,2)\nWest 2\nNorth 5\nEast 1\nSouth 3\nOptions:\nA) (0,6)\nB) (2,4)\nC) (3,2)\nD) (6,1)\nAnswer:",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,2)\nWest 2\nNorth 5\nEast 1\nSouth 3\nOptions:\nA) (0,6)\nB) (2,4)\nC) (3,2)\nD) (6,1)\nAnswer:",
    "answer": "B"
  },
  {
    "id": 13,
    "task": "formal_fallacies",
    "teacher_prompt": "Task: Decide whether the reasoning is Valid or Invalid.\n\nExample 1\nAll roses are flowers.\nThis is a rose.\nTherefore this is a flower.\nAnswer: Valid\n\nExample 2\nAll fish swim.\nNemo swims.\nTherefore Nemo is a fish.\nAnswer: Invalid\n\nExample 3\nAll students study.\nJohn studies.\nTherefore John is a student.\nAnswer: Invalid\n\nExample 4\nAll doctors studied medicine.\nSarah is a doctor.\nTherefore Sarah studied medicine.\nAnswer: Valid\n\nQuestion\nAll cars have wheels.\nThis vehicle has wheels.\nTherefore this vehicle is a car.\nOptions:\nA) Invalid\nB) Valid\nAnswer:",
    "student_prompt": "Task: Decide whether the reasoning is Valid or Invalid.\n\nQuestion\nAll cars have wheels.\nThis vehicle has wheels.\nTherefore this vehicle is a car.\nOptions:\nA) Invalid\nB) Valid\nAnswer:",
    "answer": "A"
  },
  {
    "id": 14,
    "task": "logical_deduction",
    "teacher_prompt": "Task: Answer Yes or No.\n\nExample 1\nEvery bird has wings.\nEvery animal with wings can flap.\nRobin is a bird.\nCan Robin flap?\nAnswer: Yes\n\nExample 2\nEvery teacher works at a school.\nEveryone working at a school has an ID card.\nAlice is a teacher.\nDoes Alice have an ID card?\nAnswer: Yes\n\nExample 3\nEvery elephant is an animal.\nEvery animal needs water.\nDumbo is an elephant.\nDoes Dumbo need water?\nAnswer: Yes\n\nExample 4\nEvery engineer knows math.\nEveryone who knows math can solve equations.\nEmma is an engineer.\nCan Emma solve equations?\nAnswer: Yes\n\nQuestion\nEvery programmer writes code.\nEveryone who writes code uses a computer.\nMike is a programmer.\nDoes Mike use a computer?\nOptions:\nA) No\nB) Yes\nAnswer:",
    "student_prompt": "Task: Answer Yes or No.\n\nQuestion\nEvery programmer writes code.\nEveryone who writes code uses a computer.\nMike is a programmer.\nDoes Mike use a computer?\nOptions:\nA) No\nB) Yes\nAnswer:",
    "answer": "B"
  },
  {
    "id": 15,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True OR False) AND (False OR True)\nAnswer: True\n\nExample 2\n(False AND True) OR False\nAnswer: False\n\nExample 3\n(True AND True) AND (False OR True)\nAnswer: True\n\nExample 4\n(False OR False) OR (False AND True)\nAnswer: False\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True\nAnswer:",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True AND (False OR True)) OR (False AND False)\nOptions:\nA) False\nB) True\nAnswer:",
    "answer": "B"
  },
  {
    "id": 16,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n((6+2)*3)-4\nAnswer: 20\n\nExample 2\n18/(3+3)\nAnswer: 3\n\nExample 3\n(15-5)*(8/2)\nAnswer: 40\n\nExample 4\n7+(18/3)-2\nAnswer: 11\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 14\nB) 24\nC) 36\nD) 9\nAnswer:",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 14\nB) 24\nC) 36\nD) 9\nAnswer:",
    "answer": "C"
  },
  {
    "id": 17,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n((()))\nAnswer: Balanced\n\nExample 2\n(()(()))\nAnswer: Balanced\n\nExample 3\n(()))(\nAnswer: Unbalanced\n\nExample 4\n((())())\nAnswer: Balanced\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer:",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(()()))\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer:",
    "answer": "A"
  },
  {
    "id": 18,
    "task": "word_sorting",
    "teacher_prompt": "Task: Sort the words alphabetically.\n\nExample 1\norange apple kiwi\nAnswer: apple kiwi orange\n\nExample 2\nzebra monkey lion\nAnswer: lion monkey zebra\n\nExample 3\ntable chair desk\nAnswer: chair desk table\n\nExample 4\nbanana grape cherry\nAnswer: banana cherry grape\n\nQuestion\nwindow computer bottle airplane\nOptions:\nA) airplane bottle computer window\nB) bottle computer window airplane\nC) window computer bottle airplane\nD) window computer bottle airplane\nAnswer:",
    "student_prompt": "Task: Sort the words alphabetically.\n\nQuestion\nwindow computer bottle airplane\nOptions:\nA) airplane bottle computer window\nB) bottle computer window airplane\nC) window computer bottle airplane\nD) window computer bottle airplane\nAnswer:",
    "answer": "A"
  },
  {
    "id": 19,
    "task": "date_understanding",
    "teacher_prompt": "Task: Answer the day-of-week question.\n\nExample 1\nToday is Monday.\n9 days later is Wednesday.\n\nExample 2\nToday is Friday.\n11 days later is Tuesday.\n\nExample 3\nToday is Sunday.\n15 days later is Monday.\n\nExample 4\nToday is Wednesday.\n20 days later is Tuesday.\n\nQuestion\nToday is Saturday.\n18 days later is\nOptions:\nA) Friday\nB) Monday\nC) Saturday\nD) Sunday\nE) Thursday\nF) Tuesday\nG) Wednesday\nAnswer:",
    "student_prompt": "Task: Answer the day-of-week question.\n\nQuestion\nToday is Saturday.\n18 days later is\nOptions:\nA) Friday\nB) Monday\nC) Saturday\nD) Sunday\nE) Thursday\nF) Tuesday\nG) Wednesday\nAnswer:",
    "answer": "E"
  },
  {
    "id": 20,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership after swaps.\n\nExample 1\nAlice has the pen.\nBob has the book.\nCharlie has the key.\nSwap Alice and Charlie.\nSwap Bob and Alice.\nWho has the pen?\nAnswer: Bob\n\nExample 2\nTom has the wallet.\nEmma has the phone.\nRyan has the watch.\nSwap Emma and Ryan.\nSwap Tom and Emma.\nWho has the phone?\nAnswer: Tom\n\nExample 3\nAmy has the ring.\nBen has the coin.\nCarl has the map.\nSwap Amy and Ben.\nSwap Ben and Carl.\nWho has the ring?\nAnswer: Carl\n\nExample 4\nJack has the camera.\nKate has the laptop.\nLuke has the tablet.\nSwap Luke and Jack.\nSwap Kate and Luke.\nWho has the laptop?\nAnswer: Jack\n\nQuestion\nJohn has the guitar.\nMary has the piano.\nPeter has the violin.\nSwap John and Mary.\nSwap Mary and Peter.\nWho has the guitar?\nOptions:\nA) Alice\nB) John\nC) Peter\nD) Sam\nAnswer:",
    "student_prompt": "Task: Track the ownership after swaps.\n\nQuestion\nJohn has the guitar.\nMary has the piano.\nPeter has the violin.\nSwap John and Mary.\nSwap Mary and Peter.\nWho has the guitar?\nOptions:\nA) Alice\nB) John\nC) Peter\nD) Sam\nAnswer:",
    "answer": "C"
  }
]"""

tasks = json.loads(raw_tasks_json)
print(f"Loaded {len(tasks)} multiple-choice tasks directly from variable.")


Loaded 20 multiple-choice tasks directly from variable.


## Step 3 — Evaluation Logic

Because the answers vary per task, we won't extract just two token probabilities. Instead, we will:
1. Format the prompt using the chat template.
2. Pass it through the model.
3. Check the highest probability token predicted at the final position.
4. Decode it and compare it to the true answer.

## Step 3 — Evaluation Logic

Because the answers vary per task, we won't extract just two token probabilities. Instead, we will:
1. Format the prompt using the chat template.
2. Pass it through the model.
3. Check the highest probability token predicted at the final position.
4. Decode it and compare it to the true answer.

In [15]:
def format_prompt(text):
    # Remove "Answer:" from the user prompt text so it isn't trapped
    text = text.strip()
    if text.endswith("Answer:"):
        text = text[:-7].strip()
        
    messages = [{"role": "user", "content": text}]
    prompt_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    # Force the assistant to start its turn with "Answer:"
    prompt_str += "Answer:"
    return prompt_str

def evaluate_prompt(prompt_text, true_answer):
    """
    Returns (predicted_text, is_correct, token_probability)
    """
    formatted_prompt = format_prompt(prompt_text)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = torch.nn.functional.softmax(logits, dim=-1)
        
    # Get the single most likely next token (unconstrained)
    top_token_id = torch.argmax(probs, dim=-1).item()
    top_token_prob = probs[0, top_token_id].item()
    predicted_text = tokenizer.decode([top_token_id]).strip()
    
    # Compare with true answer
    is_correct = predicted_text.upper() == str(true_answer).strip().upper()
    
    return predicted_text, is_correct, top_token_prob


## Step 4 — Run Feasibility Check

In [16]:
results = []

print("=========================================================================")
print(f"{'ID':<3} | {'Task Type':<25} | {'True Ans':<10} | {'0-Shot (Student)':<20} | {'4-Shot (Teacher)':<20}")
print("=========================================================================")

for t in tasks:
    task_id = t["id"]
    task_type = t["task"]
    true_ans = str(t["answer"])
    
    # Evaluate 0-shot Student
    s_pred, s_correct, s_prob = evaluate_prompt(t["student_prompt"], true_ans)
    s_mark = "✅" if s_correct else "❌"
    s_str = f"{s_pred} ({s_prob:.2f}) {s_mark}"
    
    # Evaluate 4-shot Teacher
    t_pred, t_correct, t_prob = evaluate_prompt(t["teacher_prompt"], true_ans)
    t_mark = "✅" if t_correct else "❌"
    t_str = f"{t_pred} ({t_prob:.2f}) {t_mark}"
    
    results.append({
        "id": task_id,
        "task": task_type,
        "teacher_correct": t_correct,
        "student_correct": s_correct,
        "teacher_pred": t_pred,
        "student_pred": s_pred,
        "teacher_prompt_full": format_prompt(t["teacher_prompt"]),
        "student_prompt_full": format_prompt(t["student_prompt"]),
    })
    
    print(f"{task_id:<3} | {task_type:<25} | {true_ans:<10} | {s_str:<20} | {t_str:<20}")

print("=========================================================================")

# Print out the exact prompts and responses for debugging
for r in results:
    print(f"\n{'='*80}")
    print(f"TASK ID: {r['id']} | TYPE: {r['task']}")
    print(f"{'-'*80}")
    print(f"TEACHER (4-Shot) PROMPT:\n{r['teacher_prompt_full']}")
    print(f"\nTEACHER PREDICTED: {r['teacher_pred']} (Correct: {r['teacher_correct']})")
    print(f"{'-'*80}")
    print(f"STUDENT (0-Shot) PROMPT:\n{r['student_prompt_full']}")
    print(f"\nSTUDENT PREDICTED: {r['student_pred']} (Correct: {r['student_correct']})")
    print(f"{'='*80}\n")


ID  | Task Type                 | True Ans   | 0-Shot (Student)     | 4-Shot (Teacher)    
1   | tracking_shuffled_objects | B          | B (0.71) ✅           | B (0.64) ✅          
2   | tracking_shuffled_objects | C          | C (0.61) ✅           | C (0.76) ✅          
3   | navigate                  | D          | B (0.41) ❌           | C (0.46) ❌          
4   | navigate                  | A          | B (0.55) ❌           | B (0.68) ❌          
5   | formal_fallacies          | A          | B (0.78) ❌           | B (0.91) ❌          
6   | logical_deduction         | B          | B (0.77) ✅           | Yes (0.78) ❌        
7   | boolean_expressions       | B          | B (0.75) ✅           | B (0.77) ✅          
8   | multistep_arithmetic      | C          | B (0.60) ❌           | B (0.28) ❌          
9   | dyck_language             | A          | B (0.55) ❌           | B (0.50) ❌          
10  | word_sorting              | A          | C (0.34) ❌           | B (0.33) ❌          

In [17]:
golden_tasks = [r for r in results if r["teacher_correct"] and not r["student_correct"]]
teacher_failures = [r for r in results if not r["teacher_correct"]]
student_successes = [r for r in results if r["student_correct"]]

print(f"Total Tasks Evaluated : {len(tasks)}")
print(f"Teacher Failures      : {len(teacher_failures)} (Teacher couldn't solve with 4 shots)")
print(f"Student Successes     : {len(student_successes)} (Student solved 0-shot; no gap to close)")
print(f"GOLDEN TASKS          : {len(golden_tasks)} (Perfect for distillation)")

if len(golden_tasks) > 0:
    print("\nGolden Task IDs:", [r["id"] for r in golden_tasks])
else:
    print("\n⚠️ No golden tasks found. We need to rethink the prompts or model size.")

Total Tasks Evaluated : 20
Teacher Failures      : 14 (Teacher couldn't solve with 4 shots)
Student Successes     : 8 (Student solved 0-shot; no gap to close)
GOLDEN TASKS          : 0 (Perfect for distillation)

⚠️ No golden tasks found. We need to rethink the prompts or model size.
